# Mock European flight network

Generate the fixed EU flight network from `infrastructure/fixtures/eu_network.py` and visualize its unique routes. Flights are aggregated into one edge per origin/destination pair so the topology stays readable while all airports remain visible.

In [ ]:
from collections import defaultdict
import socket

import networkx as nx
import setuptools

from backend.infrastructure.fixtures.eu_network import generate_network

network = generate_network()

assert len(network.airports) == 81
graph = network.to_graph()
print(f"Airports: {graph.number_of_nodes()} | Flights: {graph.number_of_edges()}")

In [ ]:
# Collapse parallel flight edges into one directed edge per route.
route_graph = nx.DiGraph(seed=network.seed, synthetic=True)

for code, attributes in graph.nodes(data=True):
    route_graph.add_node(
        code,
        label=f"{attributes['city']} ({code})",
        latitude=attributes["latitude"],
        longitude=attributes["longitude"],
        size=8,
    )

route_prices = defaultdict(list)
for origin, destination, attributes in graph.edges(data=True):
    route_prices[(origin, destination)].append(attributes["price"])

for (origin, destination), prices in route_prices.items():
    route_graph.add_edge(
        origin,
        destination,
        flights=len(prices),
        average_price=round(sum(prices) / len(prices), 2),
    )

print(
    f"Routes: {route_graph.number_of_edges()} | "
    f"Strongly connected: {nx.is_strongly_connected(route_graph)}"
)

In [ ]:
from netwulf import visualize

# Netwulf expects a node-link dictionary with `nodes` and `links` keys.
dataset = {
    "nodes": [
        {"id": code, **attributes}
        for code, attributes in route_graph.nodes(data=True)
    ],
    "links": [
        {
            "source": origin,
            "target": destination,
            "weight": attributes["flights"],
            "average_price": attributes["average_price"],
        }
        for origin, destination, attributes in route_graph.edges(data=True)
        ],
}

with socket.socket() as port_socket:
    port_socket.bind(("127.0.0.1", 0))
    available_port = port_socket.getsockname()[1]

visualize(dataset, port=available_port)